# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata (name and description)
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

To analyze or extract records, you should refer to record set and field `@id` attributes, as per Croissant convention.

In [ ]:
# List all record sets by their `@id`s, names, and fields
if hasattr(dataset, 'record_sets'):
    record_sets = dataset.record_sets
else:
    # Fallback for datasets where .record_sets might be called .record_set or varies
    record_sets = getattr(dataset, 'record_set', [])

print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet Name: {getattr(rs, 'name', None)}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    - {getattr(field, 'name', '')} (@id: {field.id}, dataType: {getattr(field, 'data_type', None)})")
    print()

## 3. Data Extraction
Load data from specific record sets into a DataFrame for analysis.

Replace the `record_set_ids` list and field references with the specific `@id` values you are interested in, based on the output above.

In [ ]:
# Example: Set up record set @ids from the overview (replace with actual IDs as discovered above)
# If you don't know the actual record set IDs, use the code from above to list them, then assign here.

record_set_ids = []  # e.g., ['cr:ExperimentProteins', 'cr:MetadataRecords']
# Let's auto-choose them for this notebook if available:
if 'record_sets' in locals() and record_sets:
    record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for: {record_set_id}")
    # Use the `records` generator for this record set:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for {record_set_id}")


## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps: filtering records, normalizing numeric fields, and grouping data.

Demonstrated below is an example using the first record set found with numeric columns. Please modify `numeric_field` and `group_field` to match your analysis goals and the real available field names.

All references (record sets, fields) use `@id` notation or their DataFrame column names (as output above).

In [ ]:
# Pick the first available DataFrame with numeric fields
import numpy as np

chosen_df = None
chosen_record_set_id = None
numeric_field = None
group_field = None

for rs_id, df in dataframes.items():
    # Try to find a numeric column
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        chosen_df = df.copy()
        chosen_record_set_id = rs_id
        numeric_field = numeric_cols[0]
        # Heuristic: try to find a plausible group field (e.g. category, sample, etc)
        possible_group_fields = [col for col in df.columns if 'sample' in col.lower() or 'type' in col.lower() or 'group' in col.lower()]
        if possible_group_fields:
            group_field = possible_group_fields[0]
        break

if chosen_df is not None and numeric_field is not None:
    print(f"Selected DataFrame: {chosen_record_set_id}\nColumns: {chosen_df.columns.tolist()}")
    print(f"Using '{numeric_field}' as the numeric field.")
    threshold = chosen_df[numeric_field].mean() if not pd.isnull(chosen_df[numeric_field].mean()) else 0
    filtered_df = chosen_df[chosen_df[numeric_field] > threshold].copy()
    print(f"Number of records where {numeric_field} > {threshold:.2f}: {len(filtered_df)}")
    
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() != 0 else 1)
    )
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped mean statistics by '{group_field}':")
        print(grouped_df.head())
else:
    print("No suitable DataFrame with numeric fields found for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and relationships by group (if available).

If you wish to visualize other fields or record sets, update the variables accordingly.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if chosen_df is not None and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(chosen_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None and group_field in chosen_df.columns:
        plt.figure(figsize=(12,6))
        sns.boxplot(data=chosen_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable DataFrame or numeric field for visualization.")

## 6. Conclusion
In this notebook, you've loaded a dataset described by a Croissant schema, explored its record sets and fields using their `@id`s, and performed basic data analysis and visualization using `mlcroissant` and pandas.

- All references to data entities (record sets, fields, columns) were made using their unique `@id` where relevant.
- The EDA and visualization steps showcased typical exploration patterns, but you are encouraged to customize field selection and analysis for your domain questions.

For more advanced processing, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/) and extend this notebook to your scientific needs.
